# 8.5 - RAG with LangChain

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook rebuilds the from-scratch RAG pipeline of Lesson 8.1 using LangChain 1.0, then adds the scaffolding a production system needs: LCEL pipes, document loaders, conversational memory, a self-corrective LangGraph loop, a tool-using agent, streaming/caching/fallbacks, LangSmith tracing, a one-object hybrid retriever, and multilingual embeddings. Every stage that was hand-built in earlier lessons becomes a single swappable component here.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - reproducibility

A one-line marker cell. Several later cells lean on seeded or deterministic behaviour (`temperature=0` on every model call), so this note flags that answers should be stable run to run.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A comment-only cell - it runs nothing and imports nothing. It exists to signal intent: the lesson favours deterministic settings so you can compare your output against the printed expectations.

**How the code works - step by step**
- A single comment line, no executable statements.

**In one line:** a placeholder that documents the notebook's deterministic stance.

**What you'll see:** (no output - environment prep)

## Setup - install dependencies

The full package set for the lesson, kept commented so a local run is not disturbed. Uncomment it on Colab or any fresh environment before running the concept cells.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install langchain langchain-core langchain-community rank-bm25 langchain-google-genai langchain-text-splitters langchain-chroma chromadb langchain-huggingface sentence-transformers langgraph python-dotenv -q  # noqa


**Explanation**

Environment prep, not logic. It lists every provider and framework package the ten concept cells import - the v1 modular split means each integration lives in its own package rather than one monolithic `langchain`.

**How the code works - step by step**
- **`langchain` / `langchain-core` / `langchain-community`** - the core Runnable protocol, prompts, and community loaders/retrievers.
- **`langchain-google-genai`** - Gemini chat + embedding models.
- **`langchain-chroma`** - the Chroma vector store integration.
- **`langchain-text-splitters`** - `RecursiveCharacterTextSplitter`.
- **`langgraph`** - the state-machine runtime for the self-corrective loop and the agent.
- **`langchain-huggingface` / `sentence-transformers`** - local BGE-M3 multilingual embeddings.
- **`rank-bm25`** - the sparse half of the ensemble retriever.
- **`python-dotenv`** - load keys from a `.env` file.

**In one line:** install the modular v1 packages, one per integration.

**What you'll see:** (no output - environment prep)

## Setup - API keys

Gemini powers every model and embedding call in the notebook, so its key must be present in the environment before any concept cell runs.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep that reads a key from the OS environment (or a `.env` file) rather than hardcoding it. `setdefault` leaves an already-set key untouched and only fills a blank if none exists.

**How the code works - step by step**
- **`import os`** - access the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - ensures the variable exists; a real key from aistudio.google.com must replace the empty string for the calls to work.

**In one line:** make sure `GOOGLE_API_KEY` is set before any model call.

**What you'll see:** (no output - environment prep)

## 1 - LangChain RAG in ~15 lines

This is the whole lesson on one screen: the same five stages you hand-built in 8.1 - load, split, embed+store, retrieve, generate - now one LangChain component each, wired together with a pipe. The 15-line count only lands because you already did the 60-line version and know what each line stands in for.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - it embeds text and calls `gemini-2.5-flash`.

In [ ]:
# LANGCHAIN RAG - the 8.1 pipeline in ~15 lines
# pip install langchain langchain-google-genai langchain-chroma langchain-text-splitters chromadb
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Documents (in-memory now; step 2 shows real file/PDF/web loaders)
docs = [
    "Netsetos refund policy: full refund within 7 days, 50% from 8-30 days, none after 30.",
    "GenAI course: 14,999 rupees. Lifetime access, Discord, weekly live sessions, certificate.",
    "Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. Python + GCP.",
    "Prerequisites: basic Python and high-school math. No ML experience needed.",
]

# 2. Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.create_documents(docs)

# 3. Embed + store. gemini-embedding-001 (text-embedding-004 was retired 2026-01-14)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4. LLM. gemini-2.5-flash (gemini-2.0-flash reached end-of-life 2026-06-01)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 5. RAG chain (LCEL)
prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY this context. If it is not there, say you do not know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# 6. Query
for q in ["What is the refund policy?", "How much does the course cost?"]:
    print(f"Q: {q}\nA: {chain.invoke(q)}\n")

# Output:
# Q: What is the refund policy?
# A: Full refund within 7 days, 50% from 8-30 days, and no refund after 30 days.
# Q: How much does the course cost?
# A: The GenAI course costs 14,999 rupees, with lifetime access included.

**Explanation**

The complete RAG pipeline as an LCEL chain. Four in-memory FAQ strings are split into chunks, embedded with `gemini-embedding-001`, stored in Chroma, and queried through a piped chain that grounds `gemini-2.5-flash` on only the retrieved context. Read it as: assemble the standard stations, then send a question down the pass.

**How the code works - step by step**
- **Imports** come from provider packages (`langchain_google_genai`, `langchain_chroma`, `langchain_text_splitters`) - the v1 modular split.
- **`docs`** - four raw FAQ strings standing in for a real corpus.
- **`RecursiveCharacterTextSplitter` + `create_documents`** - turns the strings into `Document` chunks (size 200, overlap 30).
- **`GoogleGenerativeAIEmbeddings` + `Chroma.from_documents`** - embed and store in two lines; `as_retriever(k=3)` exposes similarity search.
- **`ChatPromptTemplate`** - a context-only prompt that instructs the model to say it does not know when the answer is absent.
- **The chain** - `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()`; the dict runs both branches and feeds the prompt.
- **The loop** - `chain.invoke(q)` runs the full pipeline on each question.

**In one line:** load -> split -> embed -> store -> retrieve -> generate, connected by `|`.

**What you'll see:** For the two questions it prints grounded answers drawn only from the FAQ: a refund-policy summary (full within 7 days, 50% for 8-30 days, none after 30) and the course price (14,999 rupees with lifetime access).

## 2 - Document loaders: one import per format

The value of a loader is not that it reads a file - your 8.1 build could do that - but that *every* loader, whatever the format, returns the same `Document` shape (`page_content` + `metadata`). Swap the loader and the splitter, embedder, and chain downstream never change.

In [ ]:
# LANGCHAIN DOCUMENT LOADERS - one import per format, one uniform Document out
# pip install langchain-community pymupdf
import os, tempfile
from langchain_community.document_loaders import TextLoader

# Write a throwaway file so this block actually RUNS
path = os.path.join(tempfile.gettempdir(), "faq.txt")
with open(path, "w", encoding="utf-8") as f:
    f.write("Refund policy: full refund within 7 days.\nSupport: help@netsetos.com")

docs = TextLoader(path).load()          # -> list[Document]
print(type(docs[0]).__name__)           # Document
print(docs[0].page_content[:40])        # Refund policy: full refund within 7 days.
print(docs[0].metadata)                 # the source path, auto-attached

# The catalog: every loader returns the SAME Document shape
catalog = {
    "TextLoader": ".txt files",
    "PyMuPDFLoader": "PDF, one Document per page (+ page metadata)",
    "WebBaseLoader": "any URL -> clean text",
    "CSVLoader": "CSV -> one Document per row",
    "DirectoryLoader": "a whole folder, by glob",
}
for name, desc in catalog.items():
    print(f"{name:16s} -> {desc}")

# Output:
# Document
# Refund policy: full refund within 7 days.
# {'source': '<your-tempdir>/faq.txt'}   # actual temp path varies by OS
# TextLoader       -> .txt files
# PyMuPDFLoader    -> PDF, one Document per page (+ page metadata)
# WebBaseLoader    -> any URL -> clean text
# CSVLoader        -> CSV -> one Document per row
# DirectoryLoader  -> a whole folder, by glob

**Explanation**

A runnable demonstration of loader uniformity, not a model call. It writes a throwaway text file, loads it with `TextLoader`, and inspects the returned `Document`; then it prints a catalog of other loaders that all hand back that identical shape.

**How the code works - step by step**
- **Write a temp file** - `faq.txt` in the OS temp dir, so the block actually runs instead of being all comments.
- **`TextLoader(path).load()`** - returns `list[Document]`; the code prints the type name, the first 40 chars of `page_content`, and the auto-attached `metadata` (the source path).
- **`catalog`** - a dict mapping other loaders (`PyMuPDFLoader`, `WebBaseLoader`, `CSVLoader`, `DirectoryLoader`) to what they read; the loop prints each.

**In one line:** any format in, the same `Document` out.

**What you'll see:** Prints `Document`, the line `Refund policy: full refund within 7 days.`, the metadata dict with the temp-file source path, then five aligned catalog rows naming each loader and its input format.

## 3 - LCEL: the pipe operator and the Runnable protocol

Step 1 used the pipe without explaining it; here is the machinery. `|` builds a `RunnableSequence` where each step's output feeds the next, and every component is a Runnable exposing the same methods (`.invoke`, `.stream`, `.batch`, `.ainvoke`) - so any chain gets streaming, async, and batch for free.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - all three chains call `gemini-2.5-flash`.

In [ ]:
# LCEL - compose Runnables with the | pipe
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Chain 1: prompt | model | parser  (a RunnableSequence)
explain = ChatPromptTemplate.from_template("Explain {topic} in one sentence.") | llm | StrOutputParser()
print(explain.invoke({"topic": "RAG"}))

# Chain 2: a dict literal becomes RunnableParallel - branches run concurrently
facts = RunnableParallel(
    short=ChatPromptTemplate.from_template("Define {t} in 5 words.") | llm | StrOutputParser(),
    example=ChatPromptTemplate.from_template("Give one real example of {t}.") | llm | StrOutputParser(),
)
print(facts.invoke({"t": "an embedding"}))   # both keys computed in parallel

# Chain 3: RAG shape - retrieve | prompt | llm | parse
def retrieve(q):
    db = {"refund": "Full refund within 7 days.", "price": "14,999 rupees."}
    return next((v for k, v in db.items() if k in q.lower()), "No info found.")

rag = (
    {"context": RunnableLambda(retrieve), "question": RunnablePassthrough()}
    | ChatPromptTemplate.from_template("Context: {context}\nQ: {question}\nAnswer from context:")
    | llm | StrOutputParser()
)
print(rag.invoke("What is the refund policy?"))

# Output:
# RAG injects retrieved documents into an LLM prompt so answers are grounded in your data.
# {'short': 'A vector capturing text meaning', 'example': 'The word king as [0.21, -0.03, ...]'}
# You can get a full refund within 7 days.

**Explanation**

Three chains that show the three LCEL shapes. It is a composition demo: the same pipe operator builds a linear chain, a parallel fan-out, and a RAG-style branch-and-merge, all over one `gemini-2.5-flash` model.

**How the code works - step by step**
- **Chain 1 (`explain`)** - the minimal shape `prompt | llm | StrOutputParser()`, a `RunnableSequence`.
- **Chain 2 (`facts`)** - a `RunnableParallel` whose two keys (`short`, `example`) each run their own sub-chain concurrently, then merge into one dict.
- **`retrieve`** - a plain Python lookup function.
- **Chain 3 (`rag`)** - `RunnableLambda(retrieve)` wraps that function into a Runnable so it drops into a pipe alongside `RunnablePassthrough()`, feeding one prompt -> llm -> parser.

**In one line:** `|` composes Runnables; a dict literal forks them in parallel.

**What you'll see:** Three printed results: a one-sentence definition of RAG, a dict with `short` and `example` keys about embeddings (both branches computed in parallel), and a one-line refund answer grounded in the mini lookup.

## 4 - Conversational RAG: memory + source citations

A single-shot chain forgets everything between calls. Production chat needs two things a bare chain lacks: *memory* (so a follow-up like "refund if I do not like it?" resolves "it") and *sources* (so the answer names its document). This shows the explicit own-it-yourself version - a history list you pass each turn.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - it embeds text and calls `gemini-2.5-flash` per turn.

In [ ]:
# CONVERSATIONAL RAG - memory + source citations
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

texts = [
    "Netsetos refund: full within 7 days, 50% 8-30 days, none after 30.",
    "GenAI course: 14,999 rupees, lifetime access, Python + GCP.",
    "Prerequisites: basic Python, high-school math. No ML needed.",
]
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.create_documents(texts, metadatas=[{"source": f"doc_{i}"} for i in range(len(texts))])
vs = Chroma.from_documents(docs, GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))
retriever = vs.as_retriever(search_kwargs={"k": 2})
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer from context ONLY. If missing, say so.\n\nContext: {context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

history = []          # the running ticket (step 6 uses a checkpointer instead)
for q in ["What are the prerequisites?", "How much does it cost?", "Refund if I do not like it?"]:
    hits = retriever.invoke(q)
    context = "\n".join(d.page_content for d in hits)
    sources = [d.metadata["source"] for d in hits]
    answer = llm.invoke(prompt.format_messages(context=context, history=history, question=q)).content
    history += [HumanMessage(content=q), AIMessage(content=answer)]   # append the turn
    print(f"Q: {q}\nA: {answer.strip()[:90]}\nsources: {sources}\n")

# Output:
# Q: What are the prerequisites?  A: Basic Python and high-school math...  sources: ['doc_2']
# Q: How much does it cost?       A: 14,999 rupees with lifetime access.   sources: ['doc_1']
# Q: Refund if I do not like it?  A: Yes - full refund within 7 days...    sources: ['doc_0']

**Explanation**

A three-turn chat loop with manual memory. It builds a Chroma store whose documents carry `source` metadata, then threads a growing message history through a `MessagesPlaceholder` on every turn so the model always sees the whole conversation and can cite where each answer came from.

**How the code works - step by step**
- **Build the store** - split three texts and attach `{"source": "doc_i"}` metadata to each so retrieved docs are citable; retriever set to `k=2`.
- **`ChatPromptTemplate.from_messages`** - a system message with `{context}`, a `MessagesPlaceholder("history")` slot, and the human `{question}`.
- **`history = []`** - the running ticket; the loop retrieves for each question, formats the prompt with context + history, calls the LLM, then appends a `HumanMessage`/`AIMessage` pair.
- **`sources`** - pulled from each hit's `metadata["source"]` and printed alongside the answer.

**In one line:** append every turn to a history list so later questions resolve against earlier context.

**What you'll see:** Three Q/A blocks: prerequisites (cited to doc_2), price of 14,999 rupees (doc_1), and a refund answer for the pronoun follow-up (doc_0) - each printed with its truncated answer and source list.

## 5 - LangGraph: state machines for self-corrective RAG

LCEL is a straight line; production RAG has loops. The common one: retrieve, grade whether the docs are relevant, and if not, rewrite the query and retry. You cannot express "go back and try again" in a pipe. LangGraph models the app as a state machine so a retry becomes an *edge*, not a hack.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - the `rewrite` and `generate` nodes call `gemini-2.5-flash`.

In [ ]:
# SELF-CORRECTIVE RAG WITH LANGGRAPH - retrieve, grade, rewrite-if-weak, generate
# pip install langgraph langchain-google-genai
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

class State(TypedDict):
    question: str
    query: str
    docs: list
    answer: str
    tries: int

def retrieve(state):                       # swap in your vector store's .invoke()
    corpus = {"refund": "Refund: full within 7 days.", "price": "Course: 14,999 rupees."}
    hits = [v for k, v in corpus.items() if k in state["query"].lower()]
    return {"docs": hits, "tries": state["tries"] + 1}

def grade(state) -> str:                    # the router: relevant docs? or retry?
    if state["docs"]:
        return "generate"
    return "rewrite" if state["tries"] < 2 else "generate"

def rewrite(state):
    better = llm.invoke(f"Rewrite as a short keyword query: {state['question']}").content
    return {"query": better.strip()}

def generate(state):
    ctx = "\n".join(state["docs"]) or "No context found."
    ans = llm.invoke(f"Answer from context only.\nContext: {ctx}\nQ: {state['question']}").content
    return {"answer": ans.strip()}

g = StateGraph(State)
g.add_node("retrieve", retrieve); g.add_node("rewrite", rewrite); g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_conditional_edges("retrieve", grade, {"generate": "generate", "rewrite": "rewrite"})
g.add_edge("rewrite", "retrieve")      # the loop back
g.add_edge("generate", END)
app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "u1"}}
seed = {"question": "How do I get my money back?", "query": "money back",
        "docs": [], "answer": "", "tries": 0}
print(app.invoke(seed, cfg)["answer"])

# Output:
# "money back" misses -> grade routes to rewrite -> a keyword query -> retrieve, then generate
# (once the rewrite matches the corpus) You can get a full refund within 7 days of purchase.

**Explanation**

A self-corrective RAG graph. State flows through nodes (`retrieve`, `rewrite`, `generate`), and a `grade` router returns the name of the next node - the conditional edge that lets a weak retrieval loop back and try a rewritten query, capped so it cannot cycle forever.

**How the code works - step by step**
- **`State` (`TypedDict`)** - the shared data every node reads and updates: question, query, docs, answer, and a `tries` counter.
- **`retrieve`** - looks up a tiny corpus by the current query and increments `tries`.
- **`grade` (router)** - returns `"generate"` when docs were found, else `"rewrite"` while `tries < 2`, else `"generate"` to give up gracefully.
- **`rewrite`** - asks the LLM to turn the question into a short keyword query.
- **`generate`** - answers from whatever context was retrieved.
- **Wiring** - `add_conditional_edges("retrieve", grade, ...)` is the branch; `add_edge("rewrite", "retrieve")` is the loop back; `compile(checkpointer=InMemorySaver())` persists state under a `thread_id`.

**In one line:** retrieve -> grade -> (rewrite -> retrieve) loop -> generate, with a `tries` cap.

**What you'll see:** Prints the final answer for "How do I get my money back?": the initial "money back" query misses, `grade` routes to a rewrite, the keyword query then hits the corpus, and generate returns a full-refund-within-7-days answer.

## 6 - Agents and tools: create_agent, retriever as a tool

A chain runs a path you wrote; an agent lets the LLM choose the path. Give the model a set of tools and it decides which to call per query, reads the result, and either answers or calls another. In LangChain 1.0 there is exactly one harness for this: `create_agent`.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - the agent's reasoning runs on `gemini-2.5-flash`.

In [ ]:
# AGENTIC RAG WITH create_agent - the LLM picks the tool (LangChain 1.0)
# pip install langchain langgraph langchain-google-genai
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_docs(query: str) -> str:
    """Search the Netsetos knowledge base for policies, pricing, and schedules."""
    kb = {"refund": "Full refund within 7 days.", "price": "Course: 14,999 rupees."}
    return next((v for k, v in kb.items() if k in query.lower()), "Not in the knowledge base.")

@tool
def web_search(query: str) -> str:
    """Search the public web for anything not in the knowledge base."""
    return f"[web results for '{query}']"

agent = create_agent(
    model="google_genai:gemini-2.5-flash",        # provider:model string
    tools=[search_docs, web_search],
    system_prompt="Use search_docs for Netsetos questions, web_search otherwise.",
    checkpointer=InMemorySaver(),                # memory keyed by thread_id
)

cfg = {"configurable": {"thread_id": "u1"}}
out = agent.invoke({"messages": [{"role": "user", "content": "What is the refund policy?"}]}, cfg)
print(out["messages"][-1].content)

# Output:
# The agent calls search_docs("refund") -> "Full refund within 7 days." -> final answer:
# You can get a full refund within 7 days of purchase.

**Explanation**

An agentic RAG setup where the LLM routes. Two functions are registered as tools via `@tool` (their docstrings become the descriptions the model reads), handed to `create_agent`, which then picks the right tool for a query without any router you wrote by hand.

**How the code works - step by step**
- **`@tool search_docs`** - a knowledge-base lookup; its docstring tells the LLM to use it for policies, pricing, and schedules.
- **`@tool web_search`** - a stub for anything outside the knowledge base.
- **`create_agent`** - takes a `provider:model` string (`google_genai:gemini-2.5-flash`), the tool list, a system prompt that steers routing, and an `InMemorySaver` checkpointer for memory keyed by `thread_id`.
- **`agent.invoke`** - called with a `messages` list and a thread config; the agent selects a tool, runs it, and loops until it answers.

**In one line:** describe tools, hand them to `create_agent`, and let the LLM route.

**What you'll see:** Prints the agent's final reply to "What is the refund policy?" - it chose `search_docs`, got "Full refund within 7 days", and answered that you can get a full refund within 7 days of purchase.

## 7 - Production: streaming, caching, fallbacks

A working chain is not a production service. Three things separate them, and LCEL gives all three by *wrapping* - you do not rewrite the chain. Streaming (users see tokens as they generate), caching (repeat queries skip the model), and fallbacks (a provider outage does not crash the app).

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - it streams tokens from `gemini-2.5-flash`.

In [ ]:
# PRODUCTION LCEL - streaming, caching, provider fallbacks (all by wrapping)
import asyncio
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

set_llm_cache(InMemoryCache())          # repeat calls skip the model (RedisCache in prod)

# Fallback: if the primary errors or rate-limits, fail over automatically
primary = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, max_retries=0)
backup = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
llm = primary.with_fallbacks([backup])  # any other chat model works here too

prompt = ChatPromptTemplate.from_template("Answer in one line: {q}")
chain = prompt | llm | StrOutputParser()

# Streaming: same chain, call .astream_events instead of .invoke
async def stream_answer(q):
    async for ev in chain.astream_events({"q": q}, version="v2"):
        if ev["event"] == "on_chat_model_stream":
            print(ev["data"]["chunk"].content, end="", flush=True)

import nest_asyncio; nest_asyncio.apply()  # notebooks already run an event loop
asyncio.run(stream_answer("What is RAG in one line?"))

# Output:
# (tokens arrive one at a time, then form the full sentence)
# RAG retrieves relevant documents and injects them into the prompt for grounded answers.

**Explanation**

The same style of chain from step 1, now hardened three ways with single calls. A process-wide cache short-circuits repeats, a fallback model takes over on failure, and an async event stream surfaces tokens as they arrive - none of which touch the chain's definition.

**How the code works - step by step**
- **`set_llm_cache(InMemoryCache())`** - repeat identical calls skip the model entirely (`RedisCache` in production).
- **`with_fallbacks([backup])`** - the primary (`max_retries=0` for fast failover) hands off to `gemini-2.5-flash-lite` on error or rate limit.
- **`chain = prompt | llm | StrOutputParser()`** - an ordinary LCEL chain built over the wrapped model.
- **`stream_answer`** - iterates `chain.astream_events(..., version="v2")` and prints the content of every `on_chat_model_stream` event.
- **`nest_asyncio.apply()` + `asyncio.run`** - lets the async stream run inside the notebook's existing event loop.

**In one line:** wrap the chain for cache, fallback, and streaming without rewriting it.

**What you'll see:** Tokens print one at a time and assemble into a single-line definition of RAG - that it retrieves relevant documents and injects them into the prompt for grounded answers.

## 8 - LangSmith observability: tracing, callbacks, evaluation

When a RAG answer is wrong, "why?" has three suspects: retrieval, the prompt, or the model. Without a trace you are guessing. LangSmith records every invocation as a tree of runs so you can open the exact request; this cell shows the zero-code switch, a custom callback, and a tiny eval seed.

In [ ]:
# LANGSMITH OBSERVABILITY - zero-code tracing + a custom callback + a tiny eval
import os
os.environ["LANGSMITH_TRACING"] = "true"     # plus LANGSMITH_API_KEY and LANGSMITH_PROJECT
# Every chain .invoke() is now traced automatically - no code change.

from langchain_core.callbacks import BaseCallbackHandler

class RetrievalLogger(BaseCallbackHandler):
    def on_retriever_end(self, documents, **kwargs):
        print(f"[trace] retrieved {len(documents)} docs")
    def on_llm_end(self, response, **kwargs):
        print("[trace] llm responded")

# chain.invoke(q, config={"callbacks": [RetrievalLogger()]})

# A tiny offline eval over a golden set (LangSmith's evaluate() scales this)
golden = [
    {"q": "refund window?", "must_contain": "7 days"},
    {"q": "course price?", "must_contain": "14,999"},
]
def groundedness(answer, must):        # 1.0 if the required fact is present
    return 1.0 if must in answer else 0.0

print("golden set:", len(golden), "cases")

# Output:
# golden set: 2 cases
# (with tracing on, each request appears in LangSmith as a run tree you can open)

**Explanation**

A demonstration of the three observability entry points - it makes no model call itself. Setting one env var turns on automatic tracing, a callback subclass logs retrieval/LLM events inline, and a hand-scored golden set previews the `evaluate()` pattern expanded in Lesson 8.11.

**How the code works - step by step**
- **`LANGSMITH_TRACING="true"`** - plus a key and project, every `.invoke()` is traced with no code change.
- **`RetrievalLogger(BaseCallbackHandler)`** - overrides `on_retriever_end` and `on_llm_end` to print inline metrics; the wiring line is shown commented.
- **`golden`** - two `{q, must_contain}` test cases.
- **`groundedness`** - a scorer returning 1.0 when the required fact appears in the answer, the seed of an automated regression gate.

**In one line:** flip on tracing, hook callbacks for inline metrics, and score a golden set.

**What you'll see:** Prints `golden set: 2 cases`. With tracing enabled, each real request would also appear in the LangSmith UI as an openable run tree (nothing else prints here).

## 9 - Advanced retrievers: EnsembleRetriever

Remember the hybrid stack you hand-coded in 8.4 - BM25 for exact tokens, dense for meaning, fused with RRF? In LangChain that entire stack is one object: `EnsembleRetriever`. You get it in a line, plus a family of other retrievers for specific problems.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - the dense half embeds with `gemini-embedding-001`.

In [ ]:
# ENSEMBLE RETRIEVER - 8.4's hybrid (BM25 + dense + RRF) as ONE object
# pip install langchain langchain-community rank-bm25 langchain-chroma langchain-google-genai
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

texts = [
    "Refund policy: full refund within 7 days. Error code NTS-4021 means the token expired.",
    "The GenAI course costs 14,999 rupees with lifetime access.",
    "Live classes run daily at 7 PM IST on YouTube.",
]

# Sparse half - exact tokens like NTS-4021
bm25 = BM25Retriever.from_texts(texts); bm25.k = 3

# Dense half - meaning
emb = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
dense = Chroma.from_texts(texts, emb).as_retriever(search_kwargs={"k": 3})

# Fuse with RRF under the hood; weights control the blend
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])

for d in hybrid.invoke("what does NTS-4021 mean"):
    print("-", d.page_content[:52])

# Output:
# - Refund policy: full refund within 7 days. Error code NTS-4021...
# - The GenAI course costs 14,999 rupees with lifetime access.

**Explanation**

The 8.4 hybrid search compressed into one object. It builds a sparse `BM25Retriever` and a dense Chroma retriever, then fuses them with `EnsembleRetriever` (Reciprocal Rank Fusion under the hood), so an exact error code and a semantic query both surface the right doc.

**How the code works - step by step**
- **`BM25Retriever.from_texts` (`k=3`)** - the sparse half, matching exact tokens like `NTS-4021`.
- **`Chroma.from_texts(...).as_retriever(k=3)`** - the dense half over `gemini-embedding-001`, matching meaning.
- **`EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])`** - fuses both ranked lists with RRF; `weights` tune the blend.
- **`hybrid.invoke(...)`** - queries both and prints the fused top results.

**In one line:** BM25 + dense + RRF, tuned by `weights`, in a single object.

**What you'll see:** For "what does NTS-4021 mean" it prints the fused top hits - the refund/error-code document first (surfaced by BM25's exact match) followed by the course-price document.

## 10 - Multilingual RAG: BGE-M3 embeddings

Whichever framework you pick, multilingual RAG is mostly an embedding choice. `BAAI/bge-m3` embeds 100+ languages into one shared space, so a Hindi query retrieves a Hindi document while the framework and chain stay identical.

> **Runs a local model** - downloads `BAAI/bge-m3` via sentence-transformers on first use (no API key needed for embedding; CPU is fine but slower).

In [ ]:
# MULTILINGUAL RAG - BGE-M3 embeddings handle Hindi, English, and code-switching
# pip install langchain-huggingface sentence-transformers langchain-chroma
import unicodedata
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

def nfc(s):
    return unicodedata.normalize("NFC", s)     # normalize Unicode before indexing

texts = [nfc(t) for t in [
    "रिफंड नीति: 7 दिनों के भीतर पूरा रिफंड।",     # Hindi: full refund within 7 days
    "The GenAI course costs 14,999 rupees.",
]]
emb = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")   # 100+ languages, one model
store = Chroma.from_texts(texts, emb)

for d in store.similarity_search(nfc("पैसे कब वापस मिलेंगे?"), k=1):  # "when do I get money back?"
    print(d.page_content)

# Output:
# रिफंड नीति: 7 दिनों के भीतर पूरा रिफंड।

**Explanation**

A cross-lingual retrieval demo. It normalizes text to Unicode NFC, embeds a mixed Hindi/English corpus with the local multilingual BGE-M3 model into Chroma, and runs a Hindi query - retrieval works across languages because the embeddings share one space.

**How the code works - step by step**
- **`nfc`** - normalizes every string with `unicodedata.normalize("NFC", s)` so non-Latin scripts have one canonical encoding before indexing.
- **`texts`** - a Hindi refund line and an English price line, both NFC-normalized.
- **`HuggingFaceEmbeddings(model_name="BAAI/bge-m3")`** - one local model covering 100+ languages.
- **`Chroma.from_texts`** - stores the multilingual embeddings.
- **`similarity_search(nfc("पैसे कब वापस मिलेंगे?"), k=1)`** - a Hindi "when do I get money back?" query retrieves the Hindi refund document.

**In one line:** normalize to NFC, embed with a multilingual model, and cross-lingual retrieval just works.

**What you'll see:** Prints the Hindi refund line (रिफंड नीति: 7 दिनों के भीतर पूरा रिफंड।) - the Hindi query matched the Hindi document despite the English text also being in the store.

Across ten cells you rebuilt the 8.1 pipeline in ~15 lines and then bolted on everything a hand-rolled system never gets for free: uniform loaders, LCEL composition, memory, a retry loop, an agent router, production wrappers, traces, hybrid retrieval, and multilingual reach. The through-line is that a framework earns its dependency cost through three things - the Runnable protocol (swap any component), the LangGraph runtime (loops and routing), and observability (traces and evals). Next, Lesson 8.6 builds the same pipeline the data-centric LlamaIndex way, 8.10 grows the graph and agent into a full agentic RAG system, and 8.11 turns the golden-set eval seed here into a CI gate.